# 📈 Stock and Cryptocurrency Price Prediction  
### Using Machine Learning (Random Forest) & Deep Learning (LSTM)

This notebook predicts the future prices of **stocks and cryptocurrencies** using:

- **Random Forest Regressor** (Machine Learning)
- **LSTM Neural Network** (Deep Learning)

It supports dynamic symbols such as:

**Stocks:** AAPL, TSLA, MSFT  
**Crypto:** BTC-USD, ETH-USD, DOGE-USD, SOL-USD, BNB-USD  

You can enter *any* valid symbol from Yahoo Finance.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import MinMaxScaler

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

plt.style.use("seaborn-v0_8")

In [ ]:
symbol = input("Enter stock or crypto symbol (e.g., AAPL, TSLA, BTC-USD): ").strip()
period = "5y"

print(f"Downloading data for: {symbol} ...")
data = yf.download(symbol, period=period)

data.head()

In [ ]:
# Prepare data for Random Forest
data_rf = data.copy()
data_rf["Target"] = data_rf["Close"].shift(-1)
data_rf = data_rf.dropna()

X = data_rf[["Open", "High", "Low", "Close", "Volume"]]
y = data_rf["Target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

rf = RandomForestRegressor(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

mse = mean_squared_error(y_test, rf_pred)
print(f"Random Forest MSE: {mse:.4f}")

# Predict next day's price
next_day_features = data[["Open", "High", "Low", "Close", "Volume"]].iloc[-1:].values
next_day_prediction = rf.predict(next_day_features)[0]

print(f"\nPredicted next-day price for {symbol}: {next_day_prediction:.2f}")

In [ ]:
# Prepare data for LSTM
lstm_data = data[["Close"]].values

scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(lstm_data)

sequence_length = 60  # use last 60 days to predict next day

X_lstm = []
y_lstm = []

for i in range(sequence_length, len(scaled_data)):
    X_lstm.append(scaled_data[i-sequence_length:i, 0])
    y_lstm.append(scaled_data[i, 0])

X_lstm = np.array(X_lstm)
y_lstm = np.array(y_lstm)

# Reshape for LSTM: (samples, time_steps, features)
X_lstm = np.reshape(X_lstm, (X_lstm.shape[0], X_lstm.shape[1], 1))

X_train_lstm, X_test_lstm = X_lstm[:int(0.8*len(X_lstm))], X_lstm[int(0.8*len(X_lstm)):]
y_train_lstm, y_test_lstm = y_lstm[:int(0.8*len(y_lstm))], y_lstm[int(0.8*len(y_lstm)):]

In [ ]:
# Build the LSTM model
model = Sequential()
model.add(LSTM(50, return_sequences=True, input_shape=(X_train_lstm.shape[1], 1)))
model.add(LSTM(50, return_sequences=False))
model.add(Dense(25))
model.add(Dense(1))

model.compile(optimizer="adam", loss="mean_squared_error")

# Train the model
history = model.fit(
    X_train_lstm, y_train_lstm,
    batch_size=32,
    epochs=10,
    validation_split=0.1,
    verbose=1
)

# Predict on test data
lstm_predictions = model.predict(X_test_lstm)
lstm_predictions = scaler.inverse_transform(lstm_predictions.reshape(-1, 1))

# Inverse transform actual values
actual_prices = scaler.inverse_transform(y_test_lstm.reshape(-1, 1))

In [ ]:
# Plot actual vs predicted prices
plt.figure(figsize=(12, 6))
plt.plot(actual_prices, label="Actual Price", color="blue")
plt.plot(lstm_predictions, label="LSTM Predicted Price", color="red")
plt.title(f"{symbol} — Actual vs LSTM Predicted Prices")
plt.xlabel("Days")
plt.ylabel("Price")
plt.legend()
plt.show()

In [ ]:
# Predict the next future price using last 60 days
last_60_days = scaled_data[-60:]
last_60_days = np.reshape(last_60_days, (1, 60, 1))

future_prediction = model.predict(last_60_days)
future_prediction = scaler.inverse_transform(future_prediction)[0][0]

print(f"Predicted next-day price for {symbol} (LSTM): {future_prediction:.2f}")

# ✅ Conclusion

In this project, we built a complete **Stock & Cryptocurrency Price Prediction System** using:

### 🔹 Machine Learning  
- **Random Forest Regressor**  
- Predicts next‑day price based on OHLCV features  
- Fast, interpretable, and reliable for short‑term forecasting  

### 🔹 Deep Learning  
- **LSTM Neural Network**  
- Learns long‑term patterns in price movements  
- Produces smooth and realistic predictions  
- Generates a next‑day forecast using the last 60 days of data  

### 📊 What we achieved  
- Downloaded real market data using `yfinance`  
- Preprocessed and scaled time‑series data  
- Trained ML + DL models  
- Visualized actual vs predicted prices  
- Generated next‑day predictions for any stock or crypto symbol  

### 🚀 Next Improvements (Optional)
- Add more features (RSI, MACD, moving averages)  
- Add multi‑step forecasting (predict 7, 14, 30 days ahead)  
- Deploy as a web app using Streamlit or FastAPI  
- Automate daily predictions  

---

This notebook is now **portfolio‑ready** and demonstrates skills in:

✔ Python  
✔ Data analysis  
✔ Machine learning  
✔ Deep learning  
✔ Time‑series forecasting  
✔ Financial data modeling  
✔ Jupyter Notebook workflow  

Great work!